In [1]:
# !pip install -q -U transformers datasets accelerate lm-eval pyyaml pandas matplotlib
import os, gc, json, math, random, re, time, glob
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
!pip install -q -U lm-eval
SEED, MODEL, DEV = 6304, "Qwen/Qwen3-0.6B", "cuda"
OUT = "/kaggle/working/pa"; os.makedirs(OUT, exist_ok=True)
MAX_LEN, N_TRAIN, MB, GA = 2048, 40000, 2, 16
EB = MB * GA                      # 32
UPDATES = N_TRAIN // EB           # 1250
WARMUP = 38
CKPT_AT = {312, 625, 937, 1250}
KEEP_ALL_CKPTS = False            # full-FT recovery ckpts are ~7GB each; keep only the latest
TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
DIAG_LAYERS = [0, 13, 27]

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

tok = AutoTokenizer.from_pretrained(MODEL)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def chat(q, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def load_base():  # fp16 baseline on GPU
    return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                                attn_implementation="sdpa").to(DEV)

def dirsize(p): return sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [2]:
def norm(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", s.lower())).strip()

def build_train(path=f"{OUT}/train.pt"):
    if os.path.exists(path): return torch.load(path)
    gsm_test = {norm(x["question"]) for x in load_dataset("openai/gsm8k", "main", split="test")}
    ds = load_dataset("meta-math/MetaMathQA", split="train")
    data, sk = [], dict(empty=0, leak=0, long=0)
    for ex in ds:                                   # source order, no shuffle
        q, a, oq = ex["query"], ex["response"], ex.get("original_question") or ""
        if not (q or "").strip() or not (a or "").strip(): sk["empty"] += 1; continue
        if norm(q) in gsm_test or norm(oq) in gsm_test:    sk["leak"] += 1; continue
        p = tok(chat(q), add_special_tokens=False).input_ids
        r = tok(a, add_special_tokens=False).input_ids + [IM_END]   # response + end-of-turn
        if len(p) + len(r) > MAX_LEN: sk["long"] += 1; continue
        data.append((p, r))
        if len(data) == N_TRAIN: break
    print(len(data), sk); torch.save(data, path); return data

def collate(items):
    L = max(len(p) + len(r) for p, r in items)
    ids = torch.full((len(items), L), tok.pad_token_id); att = torch.zeros_like(ids)
    lab = torch.full_like(ids, -100)
    for i, (p, r) in enumerate(items):
        n = len(p) + len(r)
        ids[i, :n] = torch.tensor(p + r); att[i, :n] = 1; lab[i, len(p):n] = torch.tensor(r)
    return ids, att, lab                            # prompt + padding masked

In [3]:
class LoRALinear(nn.Module):
    def __init__(s, base, r, alpha, p, k=0):
        super().__init__()
        s.base, s.scale, s.k = base, alpha / r, k
        dev = base.weight.device
        s.A = nn.Parameter(torch.empty(r, base.in_features, device=dev, dtype=torch.float32))
        s.B = nn.Parameter(torch.zeros(base.out_features, r, device=dev, dtype=torch.float32))
        nn.init.kaiming_uniform_(s.A, a=math.sqrt(5))          # random A, zero B
        s.drop = nn.Dropout(p)
        if k > 0:                                              # bases once, FP32, frozen buffers
            U, _, Vh = torch.linalg.svd(base.weight.data.float(), full_matrices=False)
            s.register_buffer("Uk", U[:, :k].contiguous())
            s.register_buffer("Vk", Vh[:k].T.contiguous())

    def factors(s):                                            # effective factors, no dense projectors
        B, A = s.B, s.A
        if s.k > 0:
            with torch.autocast("cuda", enabled=False):
                B = B - s.Uk @ (s.Uk.T @ B)
                A = A - (A @ s.Vk) @ s.Vk.T
        return B, A

    def forward(s, x):
        B, A = s.factors()
        return s.base(x) + F.linear(F.linear(s.drop(x), A.to(x.dtype)), B.to(x.dtype)) * s.scale

def inject(model, r, alpha, k=0):
    seed_all()                                                 # identical init for LoRA-16 / OPLoRA-16
    for p in model.parameters(): p.requires_grad_(False)
    t0 = time.time()
    for layer in model.model.layers:
        for parent in (layer.self_attn, layer.mlp):
            for name in TARGETS:
                if hasattr(parent, name):
                    setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha, 0.05, k))
    torch.cuda.synchronize(); return time.time() - t0

def make_W0():
    p = f"{OUT}/W0_down.pt"
    if not os.path.exists(p):
        b = load_base()
        torch.save({l: b.model.layers[l].mlp.down_proj.weight.float().cpu() for l in DIAG_LAYERS}, p)
        del b; gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def delta_down(model, W0):
    out = {}
    for l in DIAG_LAYERS:
        m = model.model.layers[l].mlp.down_proj
        if isinstance(m, LoRALinear):
            B, A = m.factors(); out[l] = (m.scale * (B @ A)).float().cpu()
        else:
            out[l] = (m.weight.float() - W0[l].to(DEV)).cpu()
    return out

@torch.no_grad()
def export_eval(model, path):
    """Merge into a FRESH fp16 backbone (projected factors for OPLoRA) and save for lm-eval."""
    t0 = time.time(); fresh = load_base()
    for n, m in list(model.named_modules()):
        if isinstance(m, LoRALinear):
            B, A = m.factors()
            fresh.get_submodule(n).weight.copy_((m.base.weight.float() + m.scale * (B @ A)).half())
    if not any(isinstance(m, LoRALinear) for m in model.modules()):
        fresh.load_state_dict({k: v.half() for k, v in model.state_dict().items()})
    fresh.save_pretrained(path); tok.save_pretrained(path)
    return fresh, time.time() - t0

@torch.no_grad()
def save_adapter(model, path):
    sd = {}
    for n, m in model.named_modules():
        if isinstance(m, LoRALinear):
            B, A = m.factors(); sd[n + ".B"] = B.cpu(); sd[n + ".A"] = A.cpu()   # projected factors
    torch.save(sd, path); return os.path.getsize(path)

@torch.no_grad()
def check_merge(model, merged, batch):
    ids, att, _ = batch; ids, att = ids[:, :256].to(DEV), att[:, :256].to(DEV)
    model.eval(); merged.eval()
    with torch.autocast("cuda", dtype=torch.float16):
        a = model(input_ids=ids, attention_mask=att).logits.float()
    b = merged(input_ids=ids, attention_mask=att).logits.float()
    model.train(); return (a - b).abs().max().item()

@torch.no_grad()
def displacement(model, ref):
    s = 0.
    for n, p in model.named_parameters():
        s += (p.float() - ref[n].to(p.device).float()).pow(2).sum().item()
    return math.sqrt(s)

In [4]:
from tqdm.auto import tqdm
def lr_at(u, peak):
    if u < WARMUP: return peak * (u + 1) / WARMUP
    return peak * 0.5 * (1 + math.cos(math.pi * (u - WARMUP) / (UPDATES - WARMUP)))

def loss_sum(model, ids, att, lab):
    ids, att, lab = ids.to(DEV), att.to(DEV), lab.to(DEV)
    h = model.model(input_ids=ids, attention_mask=att).last_hidden_state[:, :-1]
    tgt = lab[:, 1:]; m = tgt != -100
    return F.cross_entropy(model.lm_head(h[m]).float(), tgt[m], reduction="sum")  # only response logits

from tqdm.auto import tqdm

def free_gpu():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def train(name, method, lr, lam=0.0, r=None, alpha=None, k=0, mid_eval=False, resume=True):
    run = f"{OUT}/{name}"
    if os.path.exists(f"{run}/stats.json"): print(name, "already done"); return
    os.makedirs(run, exist_ok=True)
    data = build_train(); make_W0(); W0 = torch.load(f"{OUT}/W0_down.pt")
    seed_all(); free_gpu(); torch.cuda.reset_peak_memory_stats()
    model = ref = opt = scaler = named = params = None
    try:
        model = load_base()
        stats = dict(name=name, method=method, lr=lr, lam=lam, r=r, alpha=alpha, k=k, ckpt_bytes={})
        if method == "full":
            ref = {n: p.detach().cpu() for n, p in model.named_parameters()}   # θ0 (fp16-exact), tied counted once
            if lam > 0:
                ref = {n: v.pin_memory() for n, v in ref.items()}              # pinned CPU, streamed per update
                stats["ref_bytes"] = sum(v.numel() * 2 for v in ref.values())
            model.float()                                                      # FP32 master params
        else:
            stats["inject_s"] = inject(model, r, alpha, k)
            if k > 0:
                stats["basis_bytes"] = sum((m.Uk.numel() + m.Vk.numel()) * 4 for m in model.modules() if isinstance(m, LoRALinear))
                stats["basis_preproc_s"] = stats["inject_s"]                   # SVD dominates inject time
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.train()

        named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
        params = [p for _, p in named]; stats["trainable"] = sum(p.numel() for p in params)
        opt = torch.optim.AdamW(
            [{"params": [p for _, p in named if p.ndim >= 2], "weight_decay": 0.01},
             {"params": [p for _, p in named if p.ndim < 2],  "weight_decay": 0.0}],
            lr=lr, betas=(0.9, 0.999), eps=1e-8)
        scaler = torch.amp.GradScaler("cuda")
        probe_batch = collate(data[:2])
        start, log, train_time, pen_t, peak = 0, [], 0., 0., 0

        ck = sorted(glob.glob(f"{run}/recovery_u*.pt"), key=lambda f: int(re.search(r"recovery_u(\d+)", f).group(1)))
        if resume and ck:                                                      # full state resume
            s = torch.load(ck[-1], map_location="cpu", weights_only=False)
            for n, p in named: p.data.copy_(s["params"][n])
            opt.load_state_dict(s["opt"]); scaler.load_state_dict(s["scaler"])
            torch.set_rng_state(s["rng_cpu"]); torch.cuda.set_rng_state(s["rng_cuda"])
            start, log, train_time, pen_t = s["u"], s["log"], s["train_time"], s["pen_t"]
            del s; free_gpu(); print("resumed from update", start)
            
        def do_export(tag):
            nonlocal peak
            peak = max(peak, torch.cuda.max_memory_allocated())
            path = f"{run}/eval_{tag}"; fresh, t = export_eval(model, path)
            torch.save(delta_down(model, W0), f"{run}/dW_{tag}.pt")
            st = dict(export_s=t, eval_model_bytes=dirsize(path))
            if method == "full": st["disp_l2"] = displacement(model, ref)
            else:
                st["adapter_bytes"] = save_adapter(model, f"{run}/adapter_{tag}.pt")
                st["merge_logit_maxdiff"] = check_merge(model, fresh, probe_batch)
            stats[tag] = st; del fresh; free_gpu(); torch.cuda.reset_peak_memory_stats()

        pbar = tqdm(range(start, UPDATES), initial=start, total=UPDATES, desc=name)
        for u in pbar:
            t0 = time.time()
            cur_lr = lr_at(u, lr)
            for g in opt.param_groups: g["lr"] = cur_lr
            batch = data[u * EB:(u + 1) * EB]                                  # fixed order, no shuffle
            mbs = [collate(batch[i:i + MB]) for i in range(0, EB, MB)]
            ntok = sum(int((l[:, 1:] != -100).sum()) for _, _, l in mbs)       # valid tokens in effective batch
            tot = 0.
            for ids, att, lab in mbs:
                with torch.autocast("cuda", dtype=torch.float16):
                    ls = loss_sum(model, ids, att, lab)
                scaler.scale(ls / ntok).backward(); tot += ls.item()
            rec = dict(u=u + 1, lr=cur_lr, sft=tot / ntok)

            if lam > 0:   # L2-SP: grad of λ·½Σ(θ-θ0)² added once per update (scaled to match GradScaler)
                tp = time.time(); sc = scaler.get_scale(); om = torch.zeros((), device=DEV)
                with torch.no_grad():
                    for n, p in named:
                        d = ref[n].to(DEV, non_blocking=True).float()
                        d.neg_().add_(p.data)                                  # d = θ - θ0, single fp32 temp
                        om += 0.5 * d.pow(2).sum()
                        if p.grad is not None: p.grad.add_(d, alpha=lam * sc)
                        del d
                om = om.item(); pen_t += time.time() - tp
                rec.update(omega=om, lam_omega=lam * om, total=rec["sft"] + lam * om)

            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
            sb = scaler.get_scale(); scaler.step(opt); scaler.update()
            rec.update(gnorm=gn, skipped=bool(scaler.get_scale() < sb))        # skipped = inf/nan grads
            opt.zero_grad(set_to_none=True); train_time += time.time() - t0; log.append(rec)
            pbar.set_postfix(loss=f"{rec['sft']:.4f}", gn=f"{gn:.2f}", skip=int(rec["skipped"]))
            if (u + 1) % 25 == 0: print(name, rec)

            if mid_eval and u + 1 == 625: do_export("mid")
            if (u + 1) in CKPT_AT:
                f = f"{run}/recovery_u{u+1}.pt"
                torch.save(dict(params={n: p.detach().cpu() for n, p in named}, opt=opt.state_dict(),
                                scaler=scaler.state_dict(), rng_cpu=torch.get_rng_state(),
                                rng_cuda=torch.cuda.get_rng_state(), u=u + 1, log=log,
                                train_time=train_time, pen_t=pen_t), f)
                stats["ckpt_bytes"][u + 1] = os.path.getsize(f)
                if not KEEP_ALL_CKPTS and method == "full":
                    for old in glob.glob(f"{run}/recovery_u*.pt"):
                        if old != f: os.remove(old)
            if u + 1 == UPDATES: do_export("final")

        stats.update(train_time_s=train_time, peak_mem_gb=max(peak, torch.cuda.max_memory_allocated()) / 2**30,
                     penalty_time_s=pen_t, n_skipped=sum(r["skipped"] for r in log))
        json.dump(log, open(f"{run}/log.json", "w"))
        json.dump(stats, open(f"{run}/stats.json", "w"), indent=1)             # also acts as the "done" marker
    finally:
        # runs even on OOM/KeyboardInterrupt: drop every big reference held by this frame
        # (do_export's closure cells get cleared too, since they point at these names)
        model = ref = opt = scaler = named = params = None
        free_gpu()

In [5]:
import lm_eval
from lm_eval.models.huggingface import HFLM

CONDS = {"baseline": MODEL,
         "fullft_mid":   f"{OUT}/full_ft/eval_mid",   "fullft_final": f"{OUT}/full_ft/eval_final",
         "l2sp_final":   f"{OUT}/full_l2sp/eval_final",
         "lora4_final":  f"{OUT}/lora4/eval_final",   "lora16_final": f"{OUT}/lora16/eval_final",
         "oplora16_final": f"{OUT}/oplora16/eval_final"}

def run_eval(tag, path):
    out = f"{OUT}/results/{tag}.json"; os.makedirs(f"{OUT}/results", exist_ok=True)
    if os.path.exists(out): return
    kw = dict(random_seed=SEED, numpy_random_seed=SEED, torch_random_seed=SEED,
              fewshot_random_seed=SEED, log_samples=True)
    lm = HFLM(pretrained=path, dtype="float16", batch_size=16)

    # --- likelihood tasks (cached, so a later failure doesn't redo them) ---
    ll = f"{OUT}/results/{tag}_ll.json"
    if os.path.exists(ll):
        d = json.load(open(ll)); res1, samp1 = d["results"], d["samples"]
    else:
        r1 = lm_eval.simple_evaluate(model=lm, tasks=["hellaswag", "boolq", "arc_easy"],
                                     num_fewshot=0, apply_chat_template=False, **kw)
        res1, samp1 = r1["results"], r1["samples"]
        json.dump(dict(results=res1, samples=samp1), open(ll, "w"), default=str)

    # --- GSM8K: force non-thinking chat template regardless of lm-eval version ---
    orig = lm.tokenizer.apply_chat_template
    def no_think(*a, **k):
        k.setdefault("enable_thinking", False); return orig(*a, **k)
    lm.tokenizer.apply_chat_template = no_think
    test = lm.tokenizer.apply_chat_template([{"role": "user", "content": "hi"}], tokenize=False, add_generation_prompt=True)
    assert "<think>\n\n</think>" in test, f"thinking not disabled: {test!r}"

    r2 = lm_eval.simple_evaluate(model=lm, tasks=["gsm8k_cot"], num_fewshot=0, apply_chat_template=True,
                                 gen_kwargs="max_gen_toks=1024,temperature=0,do_sample=False", **kw)
    m = {**res1, **r2["results"]}; g = m["gsm8k_cot"]
    metrics = dict(gsm8k_flex=g["exact_match,flexible-extract"], gsm8k_strict=g["exact_match,strict-match"],
                   hs_acc=m["hellaswag"]["acc,none"], hs_accn=m["hellaswag"]["acc_norm,none"],
                   boolq=m["boolq"]["acc,none"],
                   arc_acc=m["arc_easy"]["acc,none"], arc_accn=m["arc_easy"]["acc_norm,none"])
    samples = {**samp1, **r2["samples"]}
    for s in samples["gsm8k_cot"]:   # approx truncation flag: generation hit the 1024-token cap
        s["truncated"] = len(tok(s["resps"][0][0]).input_ids) >= 1020
    metrics["gsm8k_trunc_rate"] = float(np.mean([s["truncated"] for s in samples["gsm8k_cot"]
                                                  if s.get("filter") == "flexible-extract"]))
    json.dump(dict(metrics=metrics, samples=samples), open(out, "w"), default=str)
    print(tag, {k: round(v, 4) for k, v in metrics.items()})
    del lm; gc.collect(); torch.cuda.empty_cache()
# for t, p in CONDS.items(): run_eval(t, p)

def make_table():
    rows = {t: json.load(open(f"{OUT}/results/{t}.json"))["metrics"] for t in CONDS
            if os.path.exists(f"{OUT}/results/{t}.json")}
    df = pd.DataFrame(rows).T; keep = [c for c in df.columns if c != "gsm8k_trunc_rate"]
    pct = df[keep] * 100; d = pct - pct.loc["baseline"]; d.columns = [c + "_Δpp" for c in keep]
    return pd.concat([pct.round(2), d.round(2), (df[["gsm8k_trunc_rate"]] * 100).round(1)], axis=1)

In [6]:
def hs_pre(t):
    t = t.strip().replace(" [title]", ". "); t = re.sub(r"\[.*?\]", "", t); return t.replace("  ", " ")
def gsm_prompt(q): return chat(f"Q: {q}\nA:")                      # gsm8k_cot zero-shot, chat template
def hs_prompt(ex): return hs_pre(ex["activity_label"] + ": " + ex["ctx_a"] + " " + ex["ctx_b"].capitalize())

def make_probe():   # run BEFORE looking at results; IDs saved and reused for every model
    g = load_dataset("openai/gsm8k", "main", split="test"); h = load_dataset("Rowan/hellaswag", split="validation")
    rng = np.random.RandomState(SEED)
    gi = sorted(rng.choice(len(g), 32, replace=False).tolist()); hi = sorted(rng.choice(len(h), 32, replace=False).tolist())
    json.dump(dict(gsm8k=gi, hellaswag=hi), open(f"{OUT}/probe_ids.json", "w"))
    return g, h, gi, hi

@torch.no_grad()
def cache_inputs():   # starting model's down_proj inputs at last non-pad prompt token (bs=1 => no padding), FP32
    g, h, gi, hi = make_probe()
    m = load_base().float().eval()
    prompts = {"math": [gsm_prompt(g[i]["question"]) for i in gi], "nonmath": [hs_prompt(h[i]) for i in hi]}
    store = {}
    hooks = [m.model.layers[l].mlp.down_proj.register_forward_pre_hook(
                 lambda mod, inp, l=l: store.__setitem__(l, inp[0][0, -1].detach().float().cpu()))
             for l in DIAG_LAYERS]
    Z = {(l, k): [] for l in DIAG_LAYERS for k in prompts}
    for k, ps in prompts.items():
        for p in ps:
            m(**tok(p, return_tensors="pt", add_special_tokens=False).to(DEV))
            for l in DIAG_LAYERS: Z[(l, k)].append(store[l])
    for hk in hooks: hk.remove()
    torch.save({key: torch.stack(v) for key, v in Z.items()}, f"{OUT}/Z.pt")
    del m; gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def diagnostics(K=128):
    W0 = torch.load(f"{OUT}/W0_down.pt"); Z = torch.load(f"{OUT}/Z.pt")
    files = {"fullft_mid": "full_ft/dW_mid.pt", "fullft_final": "full_ft/dW_final.pt",
             "l2sp_final": "full_l2sp/dW_final.pt", "lora4_final": "lora4/dW_final.pt",
             "lora16_final": "lora16/dW_final.pt", "oplora16_final": "oplora16/dW_final.pt"}
    rows = []
    for tag, f in files.items():
        dd = torch.load(f"{OUT}/{f}")
        for l in DIAG_LAYERS:
            W, dW = W0[l].to(DEV), dd[l].to(DEV)
            U, _, _ = torch.linalg.svd(W, full_matrices=False); Uk = U[:, :K]      # same starting-weight bases
            row = dict(cond=tag, layer=l,
                       rho_k=((Uk.T @ dW).pow(2).sum() / dW.pow(2).sum()).item(),   # ‖UkUkᵀΔW‖²/‖ΔW‖²
                       rho_chance=K / W.shape[0],                                   # what a random direction would give
                       eta=(dW.norm() / W.norm()).item())
            for kind in ("math", "nonmath"):
                z = Z[(l, kind)].to(DEV)
                R = (z @ dW.T).norm(dim=1) / ((z @ W.T).norm(dim=1) + 1e-8)
                row[f"R_{kind}_mean"], row[f"R_{kind}_med"] = R.mean().item(), R.median().item()
            rows.append(row)
    return pd.DataFrame(rows)

In [7]:
import yaml
def load_questions(path="first_plot_questions.yaml"):
    qs = [dict(id=it["id"], system=it.get("system"), prompt=it["paraphrases"][0]) for it in yaml.safe_load(open(path))]
    assert len(qs) == 24, f"expected 24 prompts, got {len(qs)}; check the yaml keys"
    return qs

@torch.no_grad()
def gen_all(tag, path, qs, bs=8):
    out = f"{OUT}/qual/{tag}.json"; os.makedirs(f"{OUT}/qual", exist_ok=True)
    if os.path.exists(out): return
    tk = AutoTokenizer.from_pretrained(path, padding_side="left")
    m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16).to(DEV).eval()
    prompts = [chat(q["prompt"], q["system"]) for q in qs]; res = {}
    for i in range(0, len(qs), bs):
        x = tk(prompts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
        y = m.generate(**x, max_new_tokens=512, do_sample=False, temperature=None, top_p=None, top_k=None)
        for j, seq in enumerate(y[:, x.input_ids.shape[1]:]):
            ids = seq.tolist()
            trunc = not any(t in ids for t in (IM_END, tk.pad_token_id))      # never hit end-of-turn
            res[qs[i + j]["id"]] = dict(text=tk.decode(seq, skip_special_tokens=True), trunc=trunc)
    json.dump(res, open(out, "w"), indent=1); del m; gc.collect(); torch.cuda.empty_cache()

def qual_md(qs, tags, subset_only=False):
    data = {t: json.load(open(f"{OUT}/qual/{t}.json")) for t in tags}
    bases = [q["id"] for q in qs if not q["id"].endswith(("_json", "_template"))]
    if subset_only: bases = [bases[0], bases[-1]]                             # first & last plain question -> 42 responses
    md = []
    for b in bases:
        for fmt, suf in (("plain", ""), ("json", "_json"), ("template", "_template")):
            md.append(f"## {b} [{fmt}]")
            for t in tags:
                r = data[t][b + suf]; md.append(f"**{t}**{' (TRUNCATED)' if r['trunc'] else ''}\n\n{r['text']}\n")
    return "\n".join(md)

# qs = load_questions()
# for t, p in CONDS.items(): gen_all(t, p, qs)
# open(f"{OUT}/qual_all_168.md","w").write(qual_md(qs, list(CONDS)))
# open(f"{OUT}/qual_annotate_42.md","w").write(qual_md(qs, list(CONDS), subset_only=True))

In [8]:
import io
def print_results(k=128, save=True):
    buf = io.StringIO()
    def P(*a): print(*a); print(*a, file=buf)
    ld = lambda p: json.load(open(p)) if os.path.exists(p) else None
    ORDER = ["baseline","fullft_mid","fullft_final","l2sp_final","lora4_final","lora16_final","oplora16_final"]
    RUNS_ = ["full_ft","full_l2sp","lora4","lora16","oplora16"]
    bench = delta = None

    # ---- 1. benchmarks ----
    P("="*100); P("1. BENCHMARKS (%, zero-shot, full splits)   primary: gsm8k_flex, hs_accn, boolq, arc_accn")
    rows = {t: d["metrics"] for t in ORDER if (d := ld(f"{OUT}/results/{t}.json"))}
    if rows:
        df = pd.DataFrame(rows).T * 100
        cols = [c for c in ["gsm8k_flex","gsm8k_strict","hs_accn","hs_acc","boolq","arc_accn","arc_acc"] if c in df]
        bench = df[cols]; P(bench.round(2).to_string())
        P("GSM8K truncation rate (%):", df["gsm8k_trunc_rate"].round(1).to_dict())
        if "baseline" in df.index:
            delta = df[cols] - df.loc["baseline", cols]
            delta["nonmath_avg"] = delta[["hs_accn","boolq","arc_accn"]].mean(axis=1)
            P("\nChange vs baseline (percentage points):"); P(delta.round(2).to_string())

    # ---- 2. learning curves / L2-SP ----
    P("\n" + "="*100); P("2. TRAINING LOSS / REGULARIZATION")
    for r in RUNS_:
        L, S = ld(f"{OUT}/{r}/log.json"), ld(f"{OUT}/{r}/stats.json")
        if not L: continue
        sft = np.array([x["sft"] for x in L])
        P(f"{r:9s} updates={len(L)}  SFT first50={sft[:50].mean():.4f}  last50={sft[-50:].mean():.4f}  "
          f"skipped={sum(x['skipped'] for x in L)}  max_gnorm={max(x['gnorm'] for x in L):.2f}")
        if "omega" in L[0]:
            for u in (312, 625, 937, 1250):
                if len(L) >= u:
                    x = L[u-1]; P(f"   u={u}: Omega={x['omega']:.3f}  lam*Omega={x['lam_omega']:.5f}  total={x['total']:.4f}  (sft={x['sft']:.4f})")
            P(f"   sqrt(2*Omega_final) = {np.sqrt(2*L[-1]['omega']):.3f}  (~ the L2-SP displacement)")
        for tag in ("mid", "final"):
            if S and tag in S and "disp_l2" in S[tag]: P(f"   displacement ||theta-theta0||_2 at {tag}: {S[tag]['disp_l2']:.3f}")

    # ---- 3. resources ----
    P("\n" + "="*100); P("3. RESOURCES")
    rs = []
    for r in RUNS_:
        S = ld(f"{OUT}/{r}/stats.json")
        if not S: continue
        f = S.get("final", {})
        rs.append(dict(run=r, trainable_M=round(S["trainable"]/1e6, 2), train_min=round(S["train_time_s"]/60, 1),
                       peak_gb=round(S["peak_mem_gb"], 2), merged_fp16_gb=round(f.get("eval_model_bytes", 0)/2**30, 3),
                       adapter_mb=round(f.get("adapter_bytes", 0)/2**20, 1) or None,
                       recovery_ckpt_gb=round(max(S.get("ckpt_bytes", {}).values(), default=0)/2**30, 2) or None,
                       skipped=S.get("n_skipped"), merge_maxdiff=f.get("merge_logit_maxdiff"), export_s=round(f.get("export_s", 0), 1)))
        ex = {q: round(S[q], 2) for q in ("ref_bytes","penalty_time_s","basis_bytes","basis_preproc_s") if q in S}
        if ex: P(f"   method-specific overhead [{r}]: {ex}")
    if rs: P(pd.DataFrame(rs).set_index("run").to_string())
    if any(x["run"] == "full_ft" for x in rs):
        P("NOTE: full_ft was trained earlier with the unchunked loss, so its peak memory is not exactly comparable to the others.")

    # ---- 4. diagnostics ----
    P("\n" + "="*100); P(f"4. DIAGNOSTICS: down_proj layers {DIAG_LAYERS}, k={k} (a random subspace would give rho~{k/1024:.3f})")
    if os.path.exists(f"{OUT}/W0_down.pt") and os.path.exists(f"{OUT}/Z.pt"):
        W0 = torch.load(f"{OUT}/W0_down.pt", weights_only=False); Z = torch.load(f"{OUT}/Z.pt", weights_only=False)
        files = {"fullft_mid": "full_ft/dW_mid.pt", "fullft_final": "full_ft/dW_final.pt", "l2sp_final": "full_l2sp/dW_final.pt",
                 "lora4_final": "lora4/dW_final.pt", "lora16_final": "lora16/dW_final.pt", "oplora16_final": "oplora16/dW_final.pt"}
        Uk, rows = {}, []
        with torch.no_grad():
            for tag, f in files.items():
                if not os.path.exists(f"{OUT}/{f}"): continue
                dd = torch.load(f"{OUT}/{f}", weights_only=False)
                for l in DIAG_LAYERS:
                    W, dW = W0[l], dd[l]
                    if l not in Uk: Uk[l] = torch.linalg.svd(W, full_matrices=False)[0][:, :k]
                    row = dict(cond=tag, layer=l, rho_k=((Uk[l].T @ dW).pow(2).sum() / dW.pow(2).sum()).item(),
                               eta=(dW.norm() / W.norm()).item())
                    for kind in ("math", "nonmath"):
                        z = Z[(l, kind)]
                        row[f"R_{kind}"] = ((z @ dW.T).norm(dim=1) / ((z @ W.T).norm(dim=1) + 1e-8)).mean().item()
                    row["R_nonmath/math"] = row["R_nonmath"] / max(row["R_math"], 1e-12)
                    rows.append(row)
        if rows:
            dg = pd.DataFrame(rows)
            P("-- per layer --"); P(dg.round(5).to_string(index=False))
            P("-- mean over layers --")
            P(dg.groupby("cond", sort=False)[["rho_k","eta","R_math","R_nonmath","R_nonmath/math"]].mean().round(5).to_string())
            if "oplora16_final" in set(dg.cond):
                P(f"OPLoRA constraint check: max rho_k = {dg[dg.cond=='oplora16_final'].rho_k.max():.2e}  (expect ~0; tiny values come from SVD differences at the rank-k boundary)")

    # ---- 5. qualitative ----
    P("\n" + "="*100); P("5. QUALITATIVE (24 prompts per condition)")
    Q = {t: q for t in ORDER if (q := ld(f"{OUT}/qual/{t}.json"))}
    if Q:
        FENCE = "`" * 3
        def jok(s):
            s = re.sub(rf"^{FENCE}(?:json)?|{FENCE}$", "", s.strip(), flags=re.M).strip()
            try: json.loads(s); return True
            except Exception:
                try: json.loads(s[s.index("{"): s.rindex("}")+1]); return True
                except Exception: return False
        def rep(s):
            w = s.split(); g = [tuple(w[i:i+4]) for i in range(max(0, len(w)-3))]
            return 1 - len(set(g)) / len(g) if g else 0.
        rows = []
        for t, q in Q.items():
            js = [v for i, v in q.items() if i.endswith("_json")]
            rows.append(dict(cond=t, n=len(q), truncated=sum(v["trunc"] for v in q.values()),
                             json_valid=f"{sum(jok(v['text']) for v in js)}/{len(js)}",
                             the_answer_is=sum("The answer is" in v["text"] for v in q.values()),
                             mean_chars=int(np.mean([len(v["text"]) for v in q.values()])),
                             repetition=round(float(np.mean([rep(v["text"]) for v in q.values()])), 3)))
        P(pd.DataFrame(rows).set_index("cond").to_string())
        P("(the_answer_is = MetaMathQA-style ending leaking into unrelated answers; repetition = share of repeated 4-grams)")
        base_ids = [i for i in next(iter(Q.values())) if not i.endswith(("_json", "_template"))]
        for b in (base_ids[0], base_ids[-1]):                       # first & last plain question -> 42 responses
            for suf in ("", "_json", "_template"):
                P(f"\n--- {b}{suf} ---")
                for t, q in Q.items():
                    v = q.get(b + suf)
                    if v: P(f"[{t}]{' [TRUNC]' if v['trunc'] else ''} {v['text'][:220]!r}")
    if save: open("/kaggle/working/results_summary.txt", "w").write(buf.getvalue())
    return dict(bench=bench, delta=delta)

In [9]:
import zipfile, shutil, urllib.request
from tqdm.auto import tqdm
from torch.utils.checkpoint import checkpoint
from IPython.display import FileLink, display

# ---------- per-run settings (change only these two lines) ----------
RUN_BASELINE = False
STAGES = ["lora4"]

def free_gpu(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def _ce_chunk(h, w, tgt): return F.cross_entropy(F.linear(h, w).float(), tgt, reduction="sum")
def loss_sum_chunked(model, ids, att, lab, chunk=512):
    ids, att, lab = ids.to(DEV), att.to(DEV), lab.to(DEV)
    h = model.model(input_ids=ids, attention_mask=att).last_hidden_state[:, :-1]
    tgt = lab[:, 1:]; m = tgt != -100
    h, t = h[m], tgt[m]; w = model.lm_head.weight; tot = h.new_zeros((), dtype=torch.float32)
    for i in range(0, h.shape[0], chunk):
        tot = tot + checkpoint(_ce_chunk, h[i:i + chunk], w, t[i:i + chunk], use_reentrant=False)
    return tot
loss_sum = loss_sum_chunked                    # must come after Cell 4

RUNS = {
    "full_ft":  dict(method="full", lr=2e-5, lam=0.0,  mid_eval=True),
    "full_l2sp":dict(method="full", lr=2e-5, lam=1e-3),
    "lora4":    dict(method="lora", lr=2e-4, r=4,  alpha=8),
    "lora16":   dict(method="lora", lr=2e-4, r=16, alpha=32),
    "oplora16": dict(method="lora", lr=2e-4, r=16, alpha=32, k=128),
}
TAGS = {"full_ft":   [("fullft_mid", "eval_mid"), ("fullft_final", "eval_final")],
        "full_l2sp": [("l2sp_final", "eval_final")], "lora4": [("lora4_final", "eval_final")],
        "lora16":    [("lora16_final", "eval_final")], "oplora16": [("oplora16_final", "eval_final")]}

# ---------- 1) restore the uploaded dataset ----------
os.makedirs(OUT, exist_ok=True)
need_ft = "full_ft" in STAGES
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True): zipfile.ZipFile(z).extractall(OUT)
src = None
for d, ds, fs in os.walk("/kaggle/input"):
    if os.path.basename(d) == "full_ft" and "eval_final" in ds: src = os.path.dirname(d); break
if src:
    print("copying from", src)
    for item in os.listdir(src):
        if item == "full_ft" and not need_ft: continue          # skip the 3 GB of weights unless needed
        s, d = os.path.join(src, item), os.path.join(OUT, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True, copy_function=shutil.copyfile)
        else: shutil.copyfile(s, d)
os.system(f"chmod -R u+w {OUT}")
print("OUT:", sorted(os.listdir(OUT)))
if need_ft:
    assert os.path.exists(f"{OUT}/full_ft/eval_final/model.safetensors") or os.path.exists(f"{OUT}/results/fullft_final.json"), \
        "Full FT not restored: dataset not attached?"

# ---------- 2) qualitative prompts ----------
qs = None
try:
    found = glob.glob("/kaggle/input/**/first_plot_questions.yaml", recursive=True)
    if found: ypath = found[0]
    else:
        ypath = "/kaggle/working/first_plot_questions.yaml"
        if not os.path.exists(ypath):
            urllib.request.urlretrieve("https://raw.githubusercontent.com/emergent-misalignment/"
                                       "emergent-misalignment/main/evaluation/first_plot_questions.yaml", ypath)
    qs = load_questions(ypath)
except Exception as e:
    print("!! qualitative prompts NOT available:", repr(e)[:150])
print("qualitative prompts:", len(qs) if qs else "none (generation skipped; weights kept)")

# ---------- 3) pack a small zip (no weights) per model ----------
def pack(name, run, tags):
    zp = f"/kaggle/working/out_{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        add = lambda p: z.write(p, os.path.relpath(p, OUT))
        for t in tags:
            for sub in ("results", "qual"):
                if os.path.exists(f"{OUT}/{sub}/{t}.json"): add(f"{OUT}/{sub}/{t}.json")
        if run:
            for d, ds, fs in os.walk(f"{OUT}/{run}"):
                ds[:] = [x for x in ds if not x.startswith("eval_")]
                for f in fs:
                    if not f.startswith("recovery_u"): add(os.path.join(d, f))
        for f in ("probe_ids.json", "W0_down.pt", "Z.pt"):
            if os.path.exists(f"{OUT}/{f}"): add(f"{OUT}/{f}")
    print(f"{zp}: {os.path.getsize(zp)/2**20:.0f} MiB"); display(FileLink(f"out_{name}.zip"))

def stage(run):
    train(run, **RUNS[run])
    free_gpu(); tags = []
    for tag, sub in TAGS[run]:
        path = f"{OUT}/{run}/{sub}"
        run_eval(tag, path); free_gpu()
        if qs: gen_all(tag, path, qs); free_gpu()
        tags.append(tag)
    for f in glob.glob(f"{OUT}/{run}/recovery_u*.pt"): os.remove(f)
    pack(run, run, tags)
    if qs:
        for _, sub in TAGS[run]: shutil.rmtree(f"{OUT}/{run}/{sub}", ignore_errors=True)

# ---------- 4) run ----------
if RUN_BASELINE:
    run_eval("baseline", MODEL); free_gpu()
    if qs: gen_all("baseline", MODEL, qs); free_gpu()
    pack("baseline", None, ["baseline"])
for run in STAGES:
    stage(run)

print_results()          # essential results go into the logs + results_summary.txt

copying from /kaggle/input/datasets/subhankashif/fullftt
OUT: ['W0_down.pt', 'Z.pt', 'probe_ids.json']
qualitative prompts: 24


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (53 outputs left out)

/kaggle/working/out_lora4.zip: 70 MiB


/kaggle/working/out_lora4.zip

1. BENCHMARKS (%, zero-shot, full splits)   primary: gsm8k_flex, hs_accn, boolq, arc_accn
             gsm8k_flex  gsm8k_strict  hs_accn  hs_acc  boolq  arc_accn  arc_acc
lora4_final       60.12           0.0    47.21   37.85  69.85      63.3    65.53
GSM8K truncation rate (%): {'lora4_final': 0.7}

2. TRAINING LOSS / REGULARIZATION
lora4     updates=1250  SFT first50=0.4985  last50=0.2116  skipped=0  max_gnorm=2.03

3. RESOURCES
   method-specific overhead [lora4]: {'penalty_time_s': 0.0}
       trainable_M  train_min  peak_gb  merged_fp16_gb  adapter_mb  recovery_ckpt_gb  skipped  merge_maxdiff  export_s
run                                                                                                                   
lora4         2.52      153.0     2.27           1.121         9.7              0.03        0        0.65625       3.6

4. DIAGNOSTICS: down_proj layers [0, 13, 27], k=128 (a random subspace would give rho~0.125)
-- per layer --
       cond  layer   rho_k     eta  R_

{'bench':              gsm8k_flex  gsm8k_strict    hs_accn     hs_acc      boolq  \
 lora4_final   60.121304           0.0  47.211711  37.851026  69.847095   
 
               arc_accn    arc_acc  
 lora4_final  63.299663  65.530303  ,
 'delta': None}

In [10]:
import zipfile, shutil, urllib.request
from tqdm.auto import tqdm
from torch.utils.checkpoint import checkpoint
from IPython.display import FileLink, display

# ---------- per-run settings (change only these two lines) ----------
RUN_BASELINE = False
STAGES = ["lora16"]

def free_gpu(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def _ce_chunk(h, w, tgt): return F.cross_entropy(F.linear(h, w).float(), tgt, reduction="sum")
def loss_sum_chunked(model, ids, att, lab, chunk=512):
    ids, att, lab = ids.to(DEV), att.to(DEV), lab.to(DEV)
    h = model.model(input_ids=ids, attention_mask=att).last_hidden_state[:, :-1]
    tgt = lab[:, 1:]; m = tgt != -100
    h, t = h[m], tgt[m]; w = model.lm_head.weight; tot = h.new_zeros((), dtype=torch.float32)
    for i in range(0, h.shape[0], chunk):
        tot = tot + checkpoint(_ce_chunk, h[i:i + chunk], w, t[i:i + chunk], use_reentrant=False)
    return tot
loss_sum = loss_sum_chunked                    # must come after Cell 4

RUNS = {
    "full_ft":  dict(method="full", lr=2e-5, lam=0.0,  mid_eval=True),
    "full_l2sp":dict(method="full", lr=2e-5, lam=1e-3),
    "lora4":    dict(method="lora", lr=2e-4, r=4,  alpha=8),
    "lora16":   dict(method="lora", lr=2e-4, r=16, alpha=32),
    "oplora16": dict(method="lora", lr=2e-4, r=16, alpha=32, k=128),
}
TAGS = {"full_ft":   [("fullft_mid", "eval_mid"), ("fullft_final", "eval_final")],
        "full_l2sp": [("l2sp_final", "eval_final")], "lora4": [("lora4_final", "eval_final")],
        "lora16":    [("lora16_final", "eval_final")], "oplora16": [("oplora16_final", "eval_final")]}

# ---------- 1) restore the uploaded dataset ----------
os.makedirs(OUT, exist_ok=True)
need_ft = "full_ft" in STAGES
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True): zipfile.ZipFile(z).extractall(OUT)
src = None
for d, ds, fs in os.walk("/kaggle/input"):
    if os.path.basename(d) == "full_ft" and "eval_final" in ds: src = os.path.dirname(d); break
if src:
    print("copying from", src)
    for item in os.listdir(src):
        if item == "full_ft" and not need_ft: continue          # skip the 3 GB of weights unless needed
        s, d = os.path.join(src, item), os.path.join(OUT, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True, copy_function=shutil.copyfile)
        else: shutil.copyfile(s, d)
os.system(f"chmod -R u+w {OUT}")
print("OUT:", sorted(os.listdir(OUT)))
if need_ft:
    assert os.path.exists(f"{OUT}/full_ft/eval_final/model.safetensors") or os.path.exists(f"{OUT}/results/fullft_final.json"), \
        "Full FT not restored: dataset not attached?"

# ---------- 2) qualitative prompts ----------
qs = None
try:
    found = glob.glob("/kaggle/input/**/first_plot_questions.yaml", recursive=True)
    if found: ypath = found[0]
    else:
        ypath = "/kaggle/working/first_plot_questions.yaml"
        if not os.path.exists(ypath):
            urllib.request.urlretrieve("https://raw.githubusercontent.com/emergent-misalignment/"
                                       "emergent-misalignment/main/evaluation/first_plot_questions.yaml", ypath)
    qs = load_questions(ypath)
except Exception as e:
    print("!! qualitative prompts NOT available:", repr(e)[:150])
print("qualitative prompts:", len(qs) if qs else "none (generation skipped; weights kept)")

# ---------- 3) pack a small zip (no weights) per model ----------
def pack(name, run, tags):
    zp = f"/kaggle/working/out_{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        add = lambda p: z.write(p, os.path.relpath(p, OUT))
        for t in tags:
            for sub in ("results", "qual"):
                if os.path.exists(f"{OUT}/{sub}/{t}.json"): add(f"{OUT}/{sub}/{t}.json")
        if run:
            for d, ds, fs in os.walk(f"{OUT}/{run}"):
                ds[:] = [x for x in ds if not x.startswith("eval_")]
                for f in fs:
                    if not f.startswith("recovery_u"): add(os.path.join(d, f))
        for f in ("probe_ids.json", "W0_down.pt", "Z.pt"):
            if os.path.exists(f"{OUT}/{f}"): add(f"{OUT}/{f}")
    print(f"{zp}: {os.path.getsize(zp)/2**20:.0f} MiB"); display(FileLink(f"out_{name}.zip"))

def stage(run):
    train(run, **RUNS[run])
    free_gpu(); tags = []
    for tag, sub in TAGS[run]:
        path = f"{OUT}/{run}/{sub}"
        run_eval(tag, path); free_gpu()
        if qs: gen_all(tag, path, qs); free_gpu()
        tags.append(tag)
    for f in glob.glob(f"{OUT}/{run}/recovery_u*.pt"): os.remove(f)
    pack(run, run, tags)
    if qs:
        for _, sub in TAGS[run]: shutil.rmtree(f"{OUT}/{run}/{sub}", ignore_errors=True)

# ---------- 4) run ----------
if RUN_BASELINE:
    run_eval("baseline", MODEL); free_gpu()
    if qs: gen_all("baseline", MODEL, qs); free_gpu()
    pack("baseline", None, ["baseline"])
for run in STAGES:
    stage(run)

print_results()          # essential results go into the logs + results_summary.txt

copying from /kaggle/input/datasets/subhankashif/fullftt
OUT: ['W0_down.pt', 'Z.pt', 'lora4', 'probe_ids.json', 'qual', 'results', 'train.pt']
qualitative prompts: 24


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

lora16:   0%|          | 0/1250 [00:00<?, ?it/s]

lora16 {'u': 25, 'lr': 0.00013157894736842105, 'sft': 0.27404774603031157, 'gnorm': 0.4048670828342438, 'skipped': False}
lora16 {'u': 50, 'lr': 0.00019995935375248606, 'sft': 0.25410153309190897, 'gnorm': 0.3017107844352722, 'skipped': False}
lora16 {'u': 75, 'lr': 0.0001995649347969019, 'sft': 0.2557473559666612, 'gnorm': 0.3142142593860626, 'skipped': False}
lora16 {'u': 100, 'lr': 0.00019875256139039902, 'sft': 0.25708213272579794, 'gnorm': 0.3012046217918396, 'skipped': False}
lora16 {'u': 125, 'lr': 0.00019752564372032657, 'sft': 0.280348811480884, 'gnorm': 0.3239173889160156, 'skipped': False}
lora16 {'u': 150, 'lr': 0.00019588933215113926, 'sft': 0.24204421858841313, 'gnorm': 0.29858827590942383, 'skipped': False}
lora16 {'u': 175, 'lr': 0.00019385049560415794, 'sft': 0.21792182658125087, 'gnorm': 0.28268009424209595, 'skipped': False}
lora16 {'u': 200, 'lr': 0.00019141769272315858, 'sft': 0.20882608887132326, 'gnorm': 0.284180223941803, 'skipped': False}
lora16 {'u': 225, 'lr'

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

/kaggle/working/out_lora16.zip: 97 MiB


/kaggle/working/out_lora16.zip

1. BENCHMARKS (%, zero-shot, full splits)   primary: gsm8k_flex, hs_accn, boolq, arc_accn
              gsm8k_flex  gsm8k_strict  hs_accn  hs_acc  boolq  arc_accn  arc_acc
lora4_final        60.12           0.0    47.21   37.85  69.85     63.30    65.53
lora16_final       61.94           0.0    47.21   37.84  71.16     63.51    65.95
GSM8K truncation rate (%): {'lora4_final': 0.7, 'lora16_final': 0.5}

2. TRAINING LOSS / REGULARIZATION
lora4     updates=1250  SFT first50=0.4985  last50=0.2116  skipped=0  max_gnorm=2.03
lora16    updates=1250  SFT first50=0.4045  last50=0.1987  skipped=0  max_gnorm=3.91

3. RESOURCES
   method-specific overhead [lora4]: {'penalty_time_s': 0.0}
   method-specific overhead [lora16]: {'penalty_time_s': 0.0}
        trainable_M  train_min  peak_gb  merged_fp16_gb  adapter_mb  recovery_ckpt_gb  skipped  merge_maxdiff  export_s
run                                                                                                                    
lora4        

{'bench':               gsm8k_flex  gsm8k_strict    hs_accn     hs_acc      boolq  \
 lora4_final    60.121304           0.0  47.211711  37.851026  69.847095   
 lora16_final   61.940864           0.0  47.211711  37.841068  71.162080   
 
                arc_accn    arc_acc  
 lora4_final   63.299663  65.530303  
 lora16_final  63.510101  65.951178  ,
 'delta': None}

In [11]:
import zipfile, shutil, urllib.request
from tqdm.auto import tqdm
from torch.utils.checkpoint import checkpoint
from IPython.display import FileLink, display

# ---------- per-run settings (change only these two lines) ----------
RUN_BASELINE = False
STAGES = ["oplora16"]

def free_gpu(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def _ce_chunk(h, w, tgt): return F.cross_entropy(F.linear(h, w).float(), tgt, reduction="sum")
def loss_sum_chunked(model, ids, att, lab, chunk=512):
    ids, att, lab = ids.to(DEV), att.to(DEV), lab.to(DEV)
    h = model.model(input_ids=ids, attention_mask=att).last_hidden_state[:, :-1]
    tgt = lab[:, 1:]; m = tgt != -100
    h, t = h[m], tgt[m]; w = model.lm_head.weight; tot = h.new_zeros((), dtype=torch.float32)
    for i in range(0, h.shape[0], chunk):
        tot = tot + checkpoint(_ce_chunk, h[i:i + chunk], w, t[i:i + chunk], use_reentrant=False)
    return tot
loss_sum = loss_sum_chunked                    # must come after Cell 4

RUNS = {
    "full_ft":  dict(method="full", lr=2e-5, lam=0.0,  mid_eval=True),
    "full_l2sp":dict(method="full", lr=2e-5, lam=1e-3),
    "lora4":    dict(method="lora", lr=2e-4, r=4,  alpha=8),
    "lora16":   dict(method="lora", lr=2e-4, r=16, alpha=32),
    "oplora16": dict(method="lora", lr=2e-4, r=16, alpha=32, k=128),
}
TAGS = {"full_ft":   [("fullft_mid", "eval_mid"), ("fullft_final", "eval_final")],
        "full_l2sp": [("l2sp_final", "eval_final")], "lora4": [("lora4_final", "eval_final")],
        "lora16":    [("lora16_final", "eval_final")], "oplora16": [("oplora16_final", "eval_final")]}

# ---------- 1) restore the uploaded dataset ----------
os.makedirs(OUT, exist_ok=True)
need_ft = "full_ft" in STAGES
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True): zipfile.ZipFile(z).extractall(OUT)
src = None
for d, ds, fs in os.walk("/kaggle/input"):
    if os.path.basename(d) == "full_ft" and "eval_final" in ds: src = os.path.dirname(d); break
if src:
    print("copying from", src)
    for item in os.listdir(src):
        if item == "full_ft" and not need_ft: continue          # skip the 3 GB of weights unless needed
        s, d = os.path.join(src, item), os.path.join(OUT, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True, copy_function=shutil.copyfile)
        else: shutil.copyfile(s, d)
os.system(f"chmod -R u+w {OUT}")
print("OUT:", sorted(os.listdir(OUT)))
if need_ft:
    assert os.path.exists(f"{OUT}/full_ft/eval_final/model.safetensors") or os.path.exists(f"{OUT}/results/fullft_final.json"), \
        "Full FT not restored: dataset not attached?"

# ---------- 2) qualitative prompts ----------
qs = None
try:
    found = glob.glob("/kaggle/input/**/first_plot_questions.yaml", recursive=True)
    if found: ypath = found[0]
    else:
        ypath = "/kaggle/working/first_plot_questions.yaml"
        if not os.path.exists(ypath):
            urllib.request.urlretrieve("https://raw.githubusercontent.com/emergent-misalignment/"
                                       "emergent-misalignment/main/evaluation/first_plot_questions.yaml", ypath)
    qs = load_questions(ypath)
except Exception as e:
    print("!! qualitative prompts NOT available:", repr(e)[:150])
print("qualitative prompts:", len(qs) if qs else "none (generation skipped; weights kept)")

# ---------- 3) pack a small zip (no weights) per model ----------
def pack(name, run, tags):
    zp = f"/kaggle/working/out_{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        add = lambda p: z.write(p, os.path.relpath(p, OUT))
        for t in tags:
            for sub in ("results", "qual"):
                if os.path.exists(f"{OUT}/{sub}/{t}.json"): add(f"{OUT}/{sub}/{t}.json")
        if run:
            for d, ds, fs in os.walk(f"{OUT}/{run}"):
                ds[:] = [x for x in ds if not x.startswith("eval_")]
                for f in fs:
                    if not f.startswith("recovery_u"): add(os.path.join(d, f))
        for f in ("probe_ids.json", "W0_down.pt", "Z.pt"):
            if os.path.exists(f"{OUT}/{f}"): add(f"{OUT}/{f}")
    print(f"{zp}: {os.path.getsize(zp)/2**20:.0f} MiB"); display(FileLink(f"out_{name}.zip"))

def stage(run):
    train(run, **RUNS[run])
    free_gpu(); tags = []
    for tag, sub in TAGS[run]:
        path = f"{OUT}/{run}/{sub}"
        run_eval(tag, path); free_gpu()
        if qs: gen_all(tag, path, qs); free_gpu()
        tags.append(tag)
    for f in glob.glob(f"{OUT}/{run}/recovery_u*.pt"): os.remove(f)
    pack(run, run, tags)
    if qs:
        for _, sub in TAGS[run]: shutil.rmtree(f"{OUT}/{run}/{sub}", ignore_errors=True)

# ---------- 4) run ----------
if RUN_BASELINE:
    run_eval("baseline", MODEL); free_gpu()
    if qs: gen_all("baseline", MODEL, qs); free_gpu()
    pack("baseline", None, ["baseline"])
for run in STAGES:
    stage(run)

print_results()          # essential results go into the logs + results_summary.txt

copying from /kaggle/input/datasets/subhankashif/fullftt
OUT: ['W0_down.pt', 'Z.pt', 'lora16', 'lora4', 'probe_ids.json', 'qual', 'results', 'train.pt']
qualitative prompts: 24


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

oplora16:   0%|          | 0/1250 [00:00<?, ?it/s]

oplora16 {'u': 25, 'lr': 0.00013157894736842105, 'sft': 0.2783099373262591, 'gnorm': 0.4160368740558624, 'skipped': False}
oplora16 {'u': 50, 'lr': 0.00019995935375248606, 'sft': 0.25583288265433796, 'gnorm': 0.2784644365310669, 'skipped': False}
oplora16 {'u': 75, 'lr': 0.0001995649347969019, 'sft': 0.2556611415841448, 'gnorm': 0.2990526258945465, 'skipped': False}
oplora16 {'u': 100, 'lr': 0.00019875256139039902, 'sft': 0.2578699196825019, 'gnorm': 0.28818970918655396, 'skipped': False}
oplora16 {'u': 125, 'lr': 0.00019752564372032657, 'sft': 0.282349526871793, 'gnorm': 0.29797127842903137, 'skipped': False}
oplora16 {'u': 150, 'lr': 0.00019588933215113926, 'sft': 0.24299944939157575, 'gnorm': 0.2726288139820099, 'skipped': False}
oplora16 {'u': 175, 'lr': 0.00019385049560415794, 'sft': 0.21838337917105535, 'gnorm': 0.2634674608707428, 'skipped': False}
oplora16 {'u': 200, 'lr': 0.00019141769272315858, 'sft': 0.21022590046772432, 'gnorm': 0.2717941105365753, 'skipped': False}
oplora1

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

/kaggle/working/out_oplora16.zip: 97 MiB


/kaggle/working/out_oplora16.zip

1. BENCHMARKS (%, zero-shot, full splits)   primary: gsm8k_flex, hs_accn, boolq, arc_accn
                gsm8k_flex  gsm8k_strict  hs_accn  hs_acc  boolq  arc_accn  arc_acc
lora4_final          60.12           0.0    47.21   37.85  69.85     63.30    65.53
lora16_final         61.94           0.0    47.21   37.84  71.16     63.51    65.95
oplora16_final       62.17           0.0    47.14   37.74  70.89     63.22    65.57
GSM8K truncation rate (%): {'lora4_final': 0.7, 'lora16_final': 0.5, 'oplora16_final': 0.5}

2. TRAINING LOSS / REGULARIZATION
lora4     updates=1250  SFT first50=0.4985  last50=0.2116  skipped=0  max_gnorm=2.03
lora16    updates=1250  SFT first50=0.4045  last50=0.1987  skipped=0  max_gnorm=3.91
oplora16  updates=1250  SFT first50=0.4085  last50=0.1997  skipped=0  max_gnorm=3.49

3. RESOURCES
   method-specific overhead [lora4]: {'penalty_time_s': 0.0}
   method-specific overhead [lora16]: {'penalty_time_s': 0.0}
   method-specific overhead [oplora16]: {'penalty_time_

{'bench':                 gsm8k_flex  gsm8k_strict    hs_accn     hs_acc      boolq  \
 lora4_final      60.121304           0.0  47.211711  37.851026  69.847095   
 lora16_final     61.940864           0.0  47.211711  37.841068  71.162080   
 oplora16_final   62.168309           0.0  47.142004  37.741486  70.886850   
 
                  arc_accn    arc_acc  
 lora4_final     63.299663  65.530303  
 lora16_final    63.510101  65.951178  
 oplora16_final  63.215488  65.572391  ,
 'delta': None}

In [12]:
import zipfile
from IPython.display import FileLink, display

# 0) free disk (resume checkpoints and stale backup are not needed any more)
for f in glob.glob(f"{OUT}/*/recovery_u*.pt"): os.remove(f)
if os.path.exists("/kaggle/working/full_ft_backup.zip"): os.remove("/kaggle/working/full_ft_backup.zip")

# 1) save tables/plots/qualitative markdown into the bundle (a step that isn't ready yet is skipped, not fatal)
os.makedirs(f"{OUT}/report", exist_ok=True)
for name, fn in [("benchmarks.csv", make_table), ("diagnostics.csv", diagnostics),
                 ("resources.csv", resource_table), ("l2sp_summary.csv", l2sp_summary)]:
    try: fn().to_csv(f"{OUT}/report/{name}"); print("saved", name)
    except Exception as e: print("skipped", name, "->", repr(e)[:100])
try: plot_losses(); print("saved loss_curves.png")
except Exception as e: print("skipped plot ->", repr(e)[:100])
try:
    qs = load_questions()
    open(f"{OUT}/qual_all_168.md", "w").write(qual_md(qs, list(CONDS)))
    open(f"{OUT}/qual_annotate_42.md", "w").write(qual_md(qs, list(CONDS), subset_only=True)); print("saved qual md")
except Exception as e: print("skipped qual md ->", repr(e)[:100])

# 2) build the zips
def make_zip(zip_path, include):
    n = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for d, _, fs in os.walk(OUT):
            for f in fs:
                p = os.path.join(d, f)
                if include(p): z.write(p, os.path.relpath(p, OUT)); n += 1
    print(f"{zip_path}: {n} files, {os.path.getsize(zip_path)/2**20:.0f} MiB")

def is_model(p): return "/eval_mid/" in p or "/eval_final/" in p

# main bundle: everything for the report (no merged models, no train.pt cache)
make_zip("/kaggle/working/pa3_report_bundle.zip", lambda p: os.path.basename(p) != "train.pt" and not is_model(p))

# merged fp16 models (~8 GB): comment this out if you only need numbers
make_zip("/kaggle/working/pa3_models.zip", is_model)

# 3) verify the main bundle
with zipfile.ZipFile("/kaggle/working/pa3_report_bundle.zip") as z: names = set(z.namelist())
need = ["results/baseline.json", "results/fullft_mid.json", "results/fullft_final.json", "results/l2sp_final.json",
        "results/lora4_final.json", "results/lora16_final.json", "results/oplora16_final.json",
        "Z.pt", "W0_down.pt", "probe_ids.json", "loss_curves.png", "report/benchmarks.csv", "report/diagnostics.csv",
        "qual_all_168.md", "qual_annotate_42.md"] + \
       [f"{r}/{f}" for r in RUNS for f in ("stats.json", "log.json", "dW_final.pt")] + ["full_ft/dW_mid.pt"] + \
       [f"qual/{t}.json" for t in CONDS]
for n in need: print("OK     " if n in names else "MISSING", n)

# 4) download links (also in the Output panel on the right)
display(FileLink("pa3_report_bundle.zip")); display(FileLink("pa3_models.zip"))

NameError: name 'resource_table' is not defined